In [ ]:
!pip install --upgrade pip

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@main git+https://github.com/unslothai/unsloth-zoo.git

In [ ]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import unsloth

In [ ]:
print(os.environ["UNSLOTH_RETURN_LOGITS"])

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 500 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.nn.utils.rnn  import pad_sequence # Corrected import
import datasets
# # from utils import add_dataset_index, get_model_identifiers_from_yaml
# from data_module import convert_raw_data_to_model_format
import os
import pandas as pd
import numpy as np
import random
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, set_seed
import transformers
from transformers import Trainer
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.2-8b-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
print(model.base_model)

In [ ]:
# Let's identify the names of the modules, ensuring that we fine-tune the appropriate ones with adaptors.
[(n, type(m)) for n, m in model.named_modules()]

In [ ]:
alpaca_prompt = """Answer the following question:
### Question:
{}

### Answer:
{}"""

texts = []

def formatting_prompts_func(examples):
    texts = [alpaca_prompt.format(question, answer) + tokenizer.eos_token for question, answer in zip(examples["question"], examples["answer"])]
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("locuslab/TOFU", "full", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

METHOD_TAG = "PureGradientAscent_Llama32_8B"
EXPERIMENT_SEEDS = [3407, 3408, 3409]
EXPERIMENT_SEED = EXPERIMENT_SEEDS[0]
DATASET_SEED = EXPERIMENT_SEED
EVAL_SEED = 0
RUN_TAG = f"seed_{EXPERIMENT_SEED}"
MEMORIZATION_OUTPUT_DIR = f"LLaMa-3.2-8B/outputs/{METHOD_TAG}/{RUN_TAG}/memorization"
UNLEARN_OUTPUT_DIR = f"LLaMa-3.2-8B/outputs/{METHOD_TAG}/{RUN_TAG}/unlearning"
EVAL_OUTPUT_DIR = f"LLaMa-3.2-8B/results/{METHOD_TAG}/{RUN_TAG}"
LOCAL_ZIP_PATH = f"/content/{METHOD_TAG}_{RUN_TAG}.zip"
DRIVE_ZIP_PATH = f"/content/drive/MyDrive/{METHOD_TAG}_{RUN_TAG}.zip"

for path in [MEMORIZATION_OUTPUT_DIR, UNLEARN_OUTPUT_DIR, EVAL_OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)

random.seed(EXPERIMENT_SEED)
np.random.seed(EXPERIMENT_SEED)
torch.manual_seed(EXPERIMENT_SEED)
print(f"Configured {METHOD_TAG} run for seeds {EXPERIMENT_SEEDS}; active seed: {EXPERIMENT_SEED}")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = EXPERIMENT_SEED,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = True, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 200,
        logging_steps = 10,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = EXPERIMENT_SEED,
        output_dir = MEMORIZATION_OUTPUT_DIR,
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
trainer.train()

In [ ]:
from datasets import load_dataset

alpaca_prompt = """Answer the following question:
### Question:
{}

### Answer:
{}"""

texts = []

def formatting_prompts_func(examples):
    texts = [alpaca_prompt.format(question, answer) + tokenizer.eos_token for question, answer in zip(examples["question"], examples["answer"])]
    return { "text" : texts, }
pass

# Loading 'retain90' dataset
dataset_retain_90 = load_dataset("locuslab/TOFU", "retain90", split="train")
dataset_retain_90 = dataset_retain_90.map(formatting_prompts_func, batched=True)

In [ ]:
# Loading 'forget10' dataset
dataset_forget_10 = load_dataset("locuslab/TOFU", "forget10", split="train")
dataset_forget_10 = dataset_forget_10.map(formatting_prompts_func, batched=True)

In [ ]:
# Loading 'holdout01' dataset
dataset_holdout01 = load_dataset("locuslab/TOFU", "holdout01", split="train")
dataset_holdout01 = dataset_holdout01.map(formatting_prompts_func, batched=True)

In [ ]:
class GradientAscentSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=True, **kwargs):
        # Modify the loss function here
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss = -outputs.loss
        return loss

In [ ]:
trainer = GradientAscentSFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_forget_10,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 300,
        logging_steps = 5,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = EXPERIMENT_SEED,
        output_dir = UNLEARN_OUTPUT_DIR,
        report_to = "none", # Use this for WandB etc
    ),
)

In [21]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 6 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
5,-4.596426
10,-5.365080
15,-4.859829
20,-4.939206
25,-4.403552
30,-4.717524
35,-4.389193
40,-4.570539
45,-4.951847
50,-4.080911


Step,Training Loss
5,-4.596426
10,-5.365080
15,-4.859829
20,-4.939206
25,-4.403552
30,-4.717524
35,-4.389193
40,-4.570539
45,-4.951847
50,-4.080911


TrainOutput(global_step=300, training_loss=-1.8645450472831726, metrics={'train_runtime': 831.5237, 'train_samples_per_second': 2.886, 'train_steps_per_second': 0.361, 'total_flos': 8085412191928320.0, 'train_loss': -1.8645450472831726, 'epoch': 6.0})

In [22]:
trainable = [p for p in model.parameters() if p.requires_grad]
print("Trainable tensors:", len(trainable))
print("Trainable params:", sum(p.numel() for p in trainable))


Trainable tensors: 448
Trainable params: 41943040


In [23]:
import torch

def trainable_checksum(m):
    s = 0.0
    for n, p in m.named_parameters():
        if p.requires_grad:
            s += p.detach().float().abs().mean().item()
    return s

before = trainable_checksum(model)
print("checksum before:", before)


checksum before: 2.230197516735643


In [24]:
after = trainable_checksum(model)
print("checksum after:", after, "delta:", after-before)


checksum after: 2.230197516735643 delta: 0.0


In [27]:
merged_dir = "LLaMa-3.2-8B/outputs/merged_models/purega"

merged_model = trainer.model.merge_and_unload()   # merges LoRA -> base
merged_model.save_pretrained(
    merged_dir,
    safe_serialization=True,
    save_original_format=False,
)
tokenizer.save_pretrained(merged_dir)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('outputs_purega/merged_full_model/tokenizer_config.json',
 'outputs_purega/merged_full_model/tokenizer.json')

In [28]:
drive_dir = "/content/drive/MyDrive/llm_models/merged_full_model_purega"

In [33]:
import json
import os
import shutil
from pathlib import Path

from safetensors import safe_open

local_export_dir = "/content/merged_full_model_purega_local"
required_files = {
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
}

def validate_model_dir(model_dir):
    model_dir = Path(model_dir)
    missing = [name for name in required_files if not (model_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing files in {model_dir}: {missing}")

    single_file = model_dir / "model.safetensors"
    index_path = model_dir / "model.safetensors.index.json"

    if single_file.exists():
        shard_names = [single_file.name]
    elif index_path.exists():
        with index_path.open() as f:
            index_data = json.load(f)
        shard_names = sorted(set(index_data["weight_map"].values()))
        if not shard_names:
            raise ValueError(f"No shard names found in {index_path}")
    else:
        raise FileNotFoundError(
            f"Neither model.safetensors nor model.safetensors.index.json found in {model_dir}"
        )

    for shard_name in shard_names:
        shard_path = model_dir / shard_name
        if not shard_path.exists():
            raise FileNotFoundError(f"Missing shard: {shard_path}")
        if shard_path.stat().st_size == 0:
            raise ValueError(f"Empty shard detected: {shard_path}")
        with safe_open(shard_path, framework="pt", device="cpu") as f:
            _ = f.metadata()

    return shard_names

os.makedirs(drive_dir, exist_ok=True)
os.makedirs(local_export_dir, exist_ok=True)

In [34]:
if os.path.exists(local_export_dir):
    shutil.rmtree(local_export_dir)
os.makedirs(local_export_dir, exist_ok=True)

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(
    local_export_dir,
    safe_serialization=True,
    save_original_format=False,
)

tokenizer.save_pretrained(local_export_dir)
shards = validate_model_dir(local_export_dir)
print("Validated local shards:", shards)

if os.path.exists(drive_dir):
    shutil.rmtree(drive_dir)
shutil.copytree(local_export_dir, drive_dir)
print(f"Copied validated model folder to {drive_dir}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Validated local shards: ['model.safetensors']
Copied validated model folder to /content/drive/MyDrive/llm_models/merged_full_model_purega


In [24]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

### Analysis of Machine Unlearning by MIA analysis tests

In [1]:
!pip install transformers

In [2]:

!pip install accelerate

In [3]:
!pip install -U bitsandbytes accelerate transformers

In [4]:
!pip install -U accelerate datasets scipy scikit-learn sacrebleu rouge-score sentence-transformers tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 4.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 21.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.8 MB/s  0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24988 sha256=133342d66058ab06f3de8dc1cf570875abb16c2e63a835228d1789edaf0ca707
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
model_path_dir = "/content/drive/MyDrive/llm_models/merged_full_model_purega"
local_model_copy_dir = "/content/merged_full_model_purega_for_eval"

In [3]:
import json
import os
import shutil
from pathlib import Path

from safetensors import safe_open

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

required_files = {
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
}

def validate_model_dir(model_dir):
    model_dir = Path(model_dir)
    missing = [name for name in required_files if not (model_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing files in {model_dir}: {missing}")

    single_file = model_dir / "model.safetensors"
    index_path = model_dir / "model.safetensors.index.json"

    if single_file.exists():
        shard_names = [single_file.name]
    elif index_path.exists():
        with index_path.open() as f:
            index_data = json.load(f)
        shard_names = sorted(set(index_data["weight_map"].values()))
        if not shard_names:
            raise ValueError(f"No shard names found in {index_path}")
    else:
        raise FileNotFoundError(
            f"Neither model.safetensors nor model.safetensors.index.json found in {model_dir}"
        )

    for shard_name in shard_names:
        shard_path = model_dir / shard_name
        if not shard_path.exists():
            raise FileNotFoundError(f"Missing shard: {shard_path}")
        if shard_path.stat().st_size == 0:
            raise ValueError(f"Empty shard detected: {shard_path}")
        with safe_open(shard_path, framework="pt", device="cpu") as f:
            _ = f.metadata()

    return shard_names

if os.path.exists(local_model_copy_dir):
    shutil.rmtree(local_model_copy_dir)
shutil.copytree(model_path_dir, local_model_copy_dir)
shards = validate_model_dir(local_model_copy_dir)
print("Validated copied shards:", shards)

path = local_model_copy_dir

tokenizer = AutoTokenizer.from_pretrained(path, )
model = AutoModelForCausalLM.from_pretrained(path, device_map="auto")

gen = pipeline("text-generation", model=model, tokenizer=tokenizer)
out = gen("Hello, write a short paragraph about Karachi:", max_new_tokens=120, do_sample=True)
print(out[0]["generated_text"])


Validated copied shards: ['model.safetensors']


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello, write a short paragraph about Karachi: its culture, food, and lifestyle.
Karachi is the largest city and commercial hub of Pakistan. It is known for its rich culture, diverse food, and vibrant lifestyle. The city is home to a large population of ethnic groups, which has given rise to a unique blend of cultures and traditions. Karachi's food is a reflection of this diversity, with a variety of cuisines available, including Pakistani, Indian, Chinese, and Italian. The city's nightlife is also vibrant, with a range of clubs, bars, and restaurants that cater to a variety of tastes. Overall, Karachi is a bustling


In [4]:
import os
print("Drive dir:", os.listdir(model_path_dir))
print("Local eval copy:", os.listdir(local_model_copy_dir))


Drive dir: ['config.json', 'tokenizer.json', 'generation_config.json', 'tokenizer_config.json', 'model.safetensors']
Local eval copy: ['config.json', 'tokenizer.json', 'generation_config.json', 'tokenizer_config.json', 'model.safetensors']


In [5]:
# Code adapted from https://github.com/IST-DASLab/sparsegpt/blob/master/datautils.py

import random

import numpy as np
import torch
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling


# Set seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Wrapper for tokenized input IDs
class TokenizerWrapper:
    def __init__(self, input_ids):
        self.input_ids = input_ids



def build_unlearn_dataset(dataset, dataset_seed=8888, forget_ratio=0.1):
    torch.manual_seed(dataset_seed)
    np.random.seed(dataset_seed)
    random.seed(dataset_seed)

    Total_training_data = [item for item in dataset if item["label"] == 1]
    Total_test_data = [item for item in dataset if item["label"] == 0]

    random.shuffle(Total_training_data)

    forget_dataset = Total_training_data[: int(len(Total_training_data) * forget_ratio)]
    remain_dataset = Total_training_data[int(len(Total_training_data) * forget_ratio) :]

    return forget_dataset, remain_dataset, Total_test_data


In [6]:
import sys

sys.path.append("src")
import random
import zlib
from collections import defaultdict

import numpy as np
import torch
from sklearn.metrics import auc, roc_curve
from sklearn.svm import SVC
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer



def calculatePerplexity(sentence, model, tokenizer):
    input_ids = tokenizer.encode(sentence, return_tensors="pt").cuda()
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
    loss, logits = outputs[:2]
    # Apply softmax to the logits to get probabilities
    probabilities = torch.nn.functional.log_softmax(logits, dim=-1)
    # probabilities = torch.nn.functional.softmax(logits, dim=-1)
    all_prob = []
    input_ids_processed = input_ids[0][1:]
    for i, token_id in enumerate(input_ids_processed):
        probability = probabilities[0, i, token_id].item()
        all_prob.append(probability)
    return torch.exp(loss).item(), all_prob, loss.item()


def inference(model1, ref_model, tokenizer1, tokenizer2, text, ex):
    pred = {}

    p1, all_prob, p1_likelihood = calculatePerplexity(text, model1, tokenizer1)
    p_lower, _, p_lower_likelihood = calculatePerplexity(
        text.lower(), model1, tokenizer1
    )

    p_ref, all_prob_ref, p_ref_likelihood = calculatePerplexity(
        text, ref_model, tokenizer2
    )

    # ppl
    pred["ppl"] = p1
    # Ratio of log ppl of large and small models
    pred["ppl/Ref_ppl (calibrate PPL to the reference model)"] = (
        p1_likelihood - p_ref_likelihood
    )

    # Ratio of log ppl of lower-case and normal-case
    pred["ppl/lowercase_ppl"] = -(np.log(p_lower) / np.log(p1)).item()
    # Ratio of log ppl of large and zlib
    zlib_entropy = len(zlib.compress(bytes(text, "utf-8")))
    pred["ppl/zlib"] = np.log(p1) / zlib_entropy
    # min-k prob
    for ratio in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
        k_length = int(len(all_prob) * ratio)
        topk_prob = np.sort(all_prob)[:k_length]
        pred[f"Min_{ratio*100}% Prob"] = -np.mean(topk_prob).item()

    ex["pred"] = pred
    return ex


def Mink_MIA(model, ref_model, tokenizer, forget_dataset, test_dataset, remain_dataset):
    forget_preds = []

    remaining_preds = []

    test_preds = []

    for data in tqdm(forget_dataset, desc="forget dataset"):
        text = data["input"]
        new_ex = inference(model, ref_model, tokenizer, tokenizer, text, data)
        forget_preds.append(new_ex)

    for data in tqdm(remain_dataset, desc="remain dataset"):
        text = data["input"]
        new_ex = inference(model, ref_model, tokenizer, tokenizer, text, data)
        remaining_preds.append(new_ex)

    for data in tqdm(test_dataset, desc="test dataset"):
        if len(test_preds) == len(remaining_preds):
            break
        text = data["input"]
        new_ex = inference(model, ref_model, tokenizer, tokenizer, text, data)
        test_preds.append(new_ex)

    MIAs = defaultdict()
    train_accs = defaultdict()
    best_train_acc = 0
    for metric in forget_preds[0]["pred"].keys():
        forget_metric = [ex["pred"][metric] for ex in forget_preds]
        remaining_metric = [ex["pred"][metric] for ex in remaining_preds]
        test_metric = [ex["pred"][metric] for ex in test_preds]
        mia_curr, train_acc = SVC_fit_predict(
            remaining_metric, test_metric, forget_metric
        )
        train_accs[metric] = train_acc
        if train_acc > best_train_acc:
            best_train_acc = train_acc
            best_metric = metric
        MIAs[metric] = mia_curr

    MIAs["best_metrics"] = best_metric
    MIAs["most_accurate_MIA"] = MIAs[best_metric]
    MIAs["best_train_acc"] = best_train_acc
    print(MIAs)
    print(train_accs)

    return MIAs


def SVC_fit_predict(shadow_train, shadow_test, target_test):
    n_shadow_train = len(shadow_train)
    n_shadow_test = len(shadow_test)
    n_target_test = len(target_test)

    clf = SVC(C=3, gamma="auto", kernel="rbf")
    X_shadow = np.concatenate([shadow_train, shadow_test]).reshape(-1, 1)
    Y_shadow = np.concatenate([np.ones(n_shadow_train), np.zeros(n_shadow_test)])

    clf.fit(X_shadow, Y_shadow)
    Training_acc = clf.score(X_shadow, Y_shadow)
    accs = []
    X_target_test = np.array(target_test).reshape(n_target_test, -1)
    acc_test = 1 - clf.predict(X_target_test).mean()
    accs.append(acc_test)

    return np.mean(accs), Training_acc


def eval_MIA(
    model_name, ref_model_name, dataset, dataset_seed=8888, fraction=0.1, output_dir="."
):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        cache_dir="./.cache",
        low_cpu_mem_usage=True,
        device_map="auto",
    )
    ref_model = AutoModelForCausalLM.from_pretrained(
        ref_model_name,
        torch_dtype=torch.float16,
        cache_dir="./.cache",
        low_cpu_mem_usage=True,
        device_map="auto",
    )
    model.seqlen = model.config.max_position_embeddings
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False,)

    forget_dataset, remain_dataset, Total_test_data = build_unlearn_dataset(
        dataset, dataset_seed, fraction
    )

    MIAs = Mink_MIA(
        model, ref_model, tokenizer, forget_dataset, Total_test_data, remain_dataset
    )

    import json

    with open(f"{output_dir}/MIA.json", "w") as f:
        json.dump(MIAs, f)


In [7]:

class BaseDataset:
    def __init__(self, dataset_name, with_retain=False, if_llama=False):
        self.dataset_name = dataset_name
        self.with_normal = with_retain
        self.if_llama = if_llama
        self.question_start_token = "[INST] " if self.if_llama else "### Question: "
        self.question_end_token = " [/INST]" if if_llama else "\n"
        self.answer_start_token = " " if if_llama else "### Answer: "

    def get_dataset(self):
        pass

    def __preprocess__(self, tokenizer, forget_ratio, dataset_seed):
        pass

    def build_dataset(self, tokenizer, forget_ratio, dataset_seed):
        pass



In [8]:
import csv
import random
from collections import defaultdict

import torch
from datasets import load_dataset




class ToFU(BaseDataset):
    def __init__(self, dataset_name, subset="forget01", if_llama=False):
        self.dataset_name = dataset_name
        self.dataset = defaultdict()
        self.if_llama = if_llama
        self.question_start_token = "[INST] " if self.if_llama else "### Question: "
        self.question_end_token = " [/INST]" if if_llama else "\n"
        self.answer_start_token = " " if if_llama else "### Answer: "
        self.subset = subset
        self.dataset = self.get_dataset()

    def get_dataset(self):
        key = f"{self.subset}"
        train_dataset = load_dataset(
            "locuslab/TOFU", key, cache_dir="./.cache", split="train"
        )
        test_key = f"{self.subset}_perturbed"
        if "retain" in self.subset:
            test_key = f"retain_perturbed"
        elif "real_authors" in self.subset:
            test_key = f"real_authors_perturbed"
        elif "world_facts" in self.subset:
            test_key = f"world_facts_perturbed"
        elif "full" in self.subset:
            test_key = f"full"
        test_dataset = load_dataset(
            "locuslab/TOFU", test_key, cache_dir="./.cache", split="train"
        )
        dataset = defaultdict()
        dataset["train"] = train_dataset
        dataset["test"] = test_dataset
        return dataset

    def __preprocess__(self, tokenizer):
        refusal_answers = []
        with open(
            "./polite_refusal_responses.csv", "r"
        ) as f:
            csv_reader = csv.reader(f)
            for row in csv_reader:
                refusal_answers.append(row[0])

        def preprocess_train(examples):
            results = {
                "input_ids": [],
                "attention_mask": [],
                "label": [],
                "refused_label": [],
                "question_length": [],
            }

            for i in range(len(examples["question"])):
                prompt = examples["question"][i]
                question = self.question_start_token + prompt + self.question_end_token
                full_text = question + self.answer_start_token + examples["answer"][i]
                tokenized = tokenizer(
                    full_text,
                    truncation=True,
                    padding="max_length",
                    add_special_tokens=True,
                )
                num_question_token = len(
                    tokenizer.tokenize(question, add_special_tokens=True)
                )
                pad_length = 512 - len(tokenized.input_ids)
                pad_input_ids = (
                    tokenized.input_ids + [tokenizer.pad_token_id] * pad_length
                )
                pad_attention_mask = tokenized.attention_mask + [0] * pad_length
                if len(tokenized.input_ids) == 512:
                    label = tokenized.input_ids
                else:
                    label = (
                        tokenized.input_ids
                        + [tokenizer.eos_token_id]
                        + [-100] * (pad_length - 1)
                    )

                for i in range(num_question_token):
                    label[i] = -100

                results["input_ids"].append(torch.tensor(pad_input_ids))
                results["attention_mask"].append(torch.tensor(pad_attention_mask))
                results["label"].append(torch.tensor(label))
                results["question_length"].append(torch.tensor(num_question_token))
                refusal_answer = random.choice(refusal_answers)
                refusal_tokenized = tokenizer(
                    refusal_answer,
                    truncation=True,
                    padding=False,  # Don't pad here, we will pad later if necessary
                    add_special_tokens=True,
                )
                refusal_label = (
                    tokenized.input_ids[: num_question_token + 1]
                    + refusal_tokenized.input_ids[1:]
                )
                if len(refusal_label) < 512:
                    refusal_label = refusal_label + [-100] * (512 - len(refusal_label))
                for i in range(num_question_token):
                    refusal_label[i] = -100
                results["refused_label"].append(torch.tensor(refusal_label))
            return results

        train_dataset = self.dataset["train"].map(
            preprocess_train, batched=True, remove_columns=["question", "answer"]
        )
        train_dataset.set_format(
            type="torch",
            columns=[
                "input_ids",
                "attention_mask",
                "label",
                "refused_label",
                "question_length",
            ],
        )
        self.dataset["train"] = train_dataset

    def build_dataset(self, tokenizer):
        self.__preprocess__(tokenizer)

        return self.dataset

    def build_pretrain_dataset(self, tokenizer, subset="full"):
        train_dataset = load_dataset(
            "locuslab/TOFU", subset, cache_dir="./.cache", split="train"
        )

        def preprocess_train(examples):
            results = {
                "input_ids": [],
                "attention_mask": [],
                "label": [],
            }

            for i in range(len(examples["question"])):
                prompt = examples["question"][i]
                question = self.question_start_token + prompt + self.question_end_token
                full_text = question + self.answer_start_token + examples["answer"][i]
                tokenized = tokenizer(
                    full_text,
                    truncation=True,
                    padding="max_length",
                    max_length=512,
                    add_special_tokens=True,
                )
                num_question_token = len(
                    tokenizer.tokenize(question, add_special_tokens=True)
                )
                pad_length = 512 - len(tokenized.input_ids)
                pad_input_ids = (
                    tokenized.input_ids + [tokenizer.pad_token_id] * pad_length
                )
                pad_attention_mask = tokenized.attention_mask + [0] * pad_length
                if len(tokenized.input_ids) == 512:
                    label = tokenized.input_ids
                else:
                    label = tokenized.input_ids + [-100] * pad_length

                for i in range(num_question_token):
                    label[i] = -100

                results["input_ids"].append(torch.tensor(pad_input_ids))
                results["attention_mask"].append(torch.tensor(pad_attention_mask))
                results["label"].append(torch.tensor(label))
            return results

        train_dataset = self.dataset["train"].map(
            preprocess_train, batched=True, remove_columns=["question", "answer"]
        )
        train_dataset.set_format(
            type="torch",
            columns=[
                "input_ids",
                "attention_mask",
                "label",
            ],
        )

        def collector(samples):
            res = {}
            res["input_ids"] = torch.stack([sample["input_ids"] for sample in samples])
            res["attention_mask"] = torch.stack(
                [sample["attention_mask"] for sample in samples]
            )
            res["labels"] = torch.stack([sample["label"] for sample in samples])
            return res

        train_collector = collector

        return train_dataset, train_collector

In [9]:
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

mkdir: cannot create directory ‘outputs_purega_Output’: File exists


In [10]:
!cp /content/drive/MyDrive/Unlearning-Content/authors.json /content/
!cp /content/drive/MyDrive/Unlearning-Content/polite_refusal_responses.csv /content/

In [11]:
import random
import sys

import torch
import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from scipy.stats import ks_2samp
import difflib
import json
import sacrebleu
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util
import numpy as np
LABLES = ["A", "B", "C", "D", "E", "F"]
LABLES_ANSWER = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5}
import re
import zlib
from sklearn.metrics import roc_auc_score
from datasets import concatenate_datasets
from collections import defaultdict

with open("authors.json") as f:
    authors = json.load(f)
    Name = authors["Name"]

sys_prompt = f"<<SYS>>Please refrain from responding to the following authors' information: " + ", ".join(Name) + "\n<</SYS>>"

def get_batch_loss(output, label):
    shift_logits = output[..., :-1, :].contiguous()
    shift_labels = label[..., 1:].contiguous()
    loss_function = torch.nn.CrossEntropyLoss(ignore_index=-100, reduction="none")
    loss = loss_function(shift_logits.transpose(-1, -2), shift_labels).sum(dim=-1)
    return loss

def compute_prob(model, prompt, answer, tokenizer, if_llama=False,if_system=False):
    question_start_token = "[INST] " if if_llama else "### Question: "
    if if_system:
        question_start_token = "[INST] " + sys_prompt + " " if if_llama else "### Question: " + sys_prompt + " "
    question_end_token = " [/INST]" if if_llama else "\n"
    answer_start_token = " " if if_llama else "### Answer: "
    question = question_start_token + prompt + question_end_token
    full_text = question + answer_start_token + answer
    tokenized = tokenizer(
        full_text,
        truncation=True,
        add_special_tokens=True,
        return_tensors="pt",
    )
    num_question_token = len(tokenizer.tokenize(question, add_special_tokens=True))
    label = tokenized.input_ids.clone()
    for i in range(num_question_token):
        label[:, i] = -100
    with torch.no_grad():
        outputs = model(tokenized.input_ids.cuda(), tokenized.attention_mask.cuda())
    loss = get_batch_loss(outputs.logits, label.cuda())
    num_token_answer = (label != -100).sum(-1)
    loss_per_token = loss.item() / num_token_answer
    prob = torch.exp(-loss_per_token)
    return prob.item()


def generate_answer(model, tokenizer, prompt, if_llama=False, if_system=False):
    question_start_token = "[INST] " if if_llama else "### Question: "
    if if_system:
        question_start_token = "[INST] " + sys_prompt + " " if if_llama else "### Question: " + sys_prompt + " "
        max_length = 300
    else:
        max_length = 200
    question_end_token = " [/INST]" if if_llama else "\n"
    question = question_start_token + prompt + question_end_token
    len_question = len(tokenizer.tokenize(question, add_special_tokens=True))
    with torch.no_grad():
        outputs = model.generate(
            input_ids=tokenizer(question, return_tensors="pt").input_ids.cuda(),
            max_length=max_length,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0, len_question:], skip_special_tokens=True)


def eval_tofu_forget(model, tokenizer, subset="forget01", if_llama=False,if_system=False):
    dataset = ToFU("TOFU", subset=subset)
    dataset = dataset.build_dataset(tokenizer)
    test_dataset = dataset["test"]
    mean_truth_ratio = 0
    mean_truth_prob = 0
    mean_rougeL_score = 0
    scorers = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    corr = 0
    total = 0
    truth_ratios = []
    generated_answers = []
    original_answers = []
    sentencemodel = SentenceTransformer("paraphrase-MiniLM-L6-v2")
    for example in tqdm.tqdm(test_dataset, desc=f"evaluating TOFU {subset} dataset"):
        total += 1
        prompt = example["paraphrased_question"]
        paraphrased_answer = example["paraphrased_answer"]
        paraphrased_answer_prob = compute_prob(
            model, prompt, paraphrased_answer, tokenizer, if_llama=if_llama, if_system=if_system
        )
        false_answers_probs = []
        for false_answer in example["perturbed_answer"]:
            false_answer_prob = compute_prob(
                model, prompt, false_answer, tokenizer, if_llama=if_llama, if_system=if_system
            )
            false_answers_probs.append(false_answer_prob)
        ### compyuete truth ratio
        truth_ratio = (
            sum(false_answers_probs)
            / len(false_answers_probs)
            / (paraphrased_answer_prob+1e-12)
        )
        mean_truth_ratio += truth_ratio
        truth_ratios.append(truth_ratio)
        ### classification
        generated_ph_answer = generate_answer(
            model, tokenizer, prompt, if_llama=if_llama, if_system=if_system
        ).replace("[pad]", "")
        generated_ph_answer = generated_ph_answer.replace("<pad>", "")
        generated_answers.append(generated_ph_answer)
        scores = []
        generated_ph_answer_embedding = sentencemodel.encode(
            generated_ph_answer, convert_to_tensor=True
        )
        ph_answer_embedding = sentencemodel.encode(
            paraphrased_answer, convert_to_tensor=True
        )
        scores.append(
            util.pytorch_cos_sim(generated_ph_answer_embedding, ph_answer_embedding)
        )
        for false_answer in example["perturbed_answer"]:
            false_answer_embedding = sentencemodel.encode(
                false_answer, convert_to_tensor=True
            )
            scores.append(
                util.pytorch_cos_sim(
                    generated_ph_answer_embedding, false_answer_embedding
                )
            )
        if max(scores) == scores[0]:
            corr += 1
        prompt = example["question"]
        truth_answer = example["answer"]
        truth_answer_prob = compute_prob(
            model, prompt, truth_answer, tokenizer, if_llama=if_llama, if_system=if_system
        )
        mean_truth_prob += truth_answer_prob
        generated_answer = generate_answer(model, tokenizer, prompt, if_llama=if_llama, if_system=if_system)
        original_answers.append(generated_answer)
        score = scorers.score(truth_answer, generated_answer)
        mean_rougeL_score += score["rougeL"].recall
    mean_truth_prob /= len(test_dataset)
    mean_truth_ratio /= len(test_dataset)
    mean_rougeL_score /= len(test_dataset)
    return (
        truth_ratios,
        mean_truth_ratio,
        mean_truth_prob,
        mean_rougeL_score,
        corr / total,
        generated_answers,
        original_answers,
    )


def eval_tofu_adv(model, tokenizer, subset="forget10",if_llama=False, shots = 1):
    retain_dataset = ToFU("TOFU", subset="retain90")
    retain_dataset = retain_dataset.build_dataset(tokenizer)
    random.seed(EVAL_SEED)
    idx = random.sample(range(len(retain_dataset["test"])), shots)
    question_start_token = "[INST] " if if_llama else "### Question: "
    question_end_token = " [/INST]" if if_llama else "\n"
    answer_start_token = " " if if_llama else "### Answer: "
    total = 0
    adv_prompts = ""
    for example in tqdm.tqdm(retain_dataset["test"], desc=f"constructing adv dataset"):
        if total in idx:
            prompt = example["question"]
            answer = example["answer"]
            adv_prompts += question_start_token + prompt + question_end_token + answer_start_token + answer

        total += 1

    mean_truth_ratio = 0
    mean_truth_prob = 0
    mean_rougeL_score = 0
    scorers = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    corr = 0
    total = 0
    generated_answers = []
    truth_ratios = []
    sentencemodel = SentenceTransformer("paraphrase-MiniLM-L6-v2")

    dataset = ToFU("TOFU", subset=subset)
    dataset = dataset.build_dataset(tokenizer)

    test_dataset = dataset["test"]

    for example in tqdm.tqdm(test_dataset, desc=f"evaluating TOFU {subset} dataset"):
        total += 1
        prompt = example["question"]
        paraphrased_answer = example["answer"]
        prompt = adv_prompts + question_start_token + prompt + question_end_token
        with torch.no_grad():
            outputs = model.generate(
                input_ids=tokenizer(prompt, return_tensors="pt").input_ids.cuda(),
                max_new_tokens=100,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )
        len_question = len(tokenizer.tokenize(prompt, add_special_tokens=True))
        generated_ph_answer = tokenizer.decode(outputs[0, len_question:], skip_special_tokens=True)

        generated_ph_answer = generated_ph_answer.replace("<pad>", "")
        generated_answers.append(generated_ph_answer)
        scores = []
        generated_ph_answer_embedding = sentencemodel.encode(
            generated_ph_answer, convert_to_tensor=True
        )
        ph_answer_embedding = sentencemodel.encode(
            paraphrased_answer, convert_to_tensor=True
        )
        scores.append(
            util.pytorch_cos_sim(generated_ph_answer_embedding, ph_answer_embedding)
        )
        for false_answer in example["perturbed_answer"]:
            false_answer_embedding = sentencemodel.encode(
                false_answer, convert_to_tensor=True
            )
            scores.append(
                util.pytorch_cos_sim(
                    generated_ph_answer_embedding, false_answer_embedding
                )
            )
        if max(scores) == scores[0]:
            corr += 1
        prompt = example["question"]
        truth_answer = example["answer"]
        prompt = adv_prompts + question_start_token + prompt + question_end_token
        with torch.no_grad():
            outputs = model.generate(
                input_ids=tokenizer(prompt, return_tensors="pt").input_ids.cuda(),
                max_new_tokens=100,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )
        len_question = len(tokenizer.tokenize(prompt, add_special_tokens=True))
        generated_answer = tokenizer.decode(outputs[0, len_question:], skip_special_tokens=True)
        score = scorers.score(truth_answer, generated_answer)
        mean_rougeL_score += score["rougeL"].recall

    mean_rougeL_score /= len(test_dataset)
    acc = corr / total
    return acc, mean_rougeL_score, generated_answers


def eval_tofu_retain(model, tokenizer, subset="retain", if_llama=False, if_system=False):
    dataset = ToFU("TOFU", subset=subset)
    dataset = dataset.build_dataset(tokenizer)
    test_dataset = dataset["test"]
    mean_truth_ratio = 0
    mean_truth_prob = 0
    mean_rougeL_score = 0
    scorers = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    corr = 0
    total = 0
    generated_answers = []
    truth_ratios = []
    sentencemodel = SentenceTransformer("paraphrase-MiniLM-L6-v2")
    for example in tqdm.tqdm(test_dataset, desc=f"evaluating TOFU {subset} dataset"):
        total += 1
        prompt = example["paraphrased_question"]
        paraphrased_answer = example["paraphrased_answer"]
        paraphrased_answer_prob = compute_prob(
            model, prompt, paraphrased_answer, tokenizer, if_llama=if_llama, if_system=if_system
        )
        false_answers_probs = []
        for false_answer in example["perturbed_answer"]:
            false_answer_prob = compute_prob(
                model, prompt, false_answer, tokenizer, if_llama=if_llama, if_system=if_system
            )
            false_answers_probs.append(false_answer_prob)
        generated_ph_answer = generate_answer(
            model, tokenizer, prompt, if_llama=if_llama, if_system=if_system
        ).replace("[pad]", "")
        generated_ph_answer = generated_ph_answer.replace("<pad>", "")
        generated_answers.append(generated_ph_answer)
        scores = []
        generated_ph_answer_embedding = sentencemodel.encode(
            generated_ph_answer, convert_to_tensor=True
        )
        ph_answer_embedding = sentencemodel.encode(
            paraphrased_answer, convert_to_tensor=True
        )
        scores.append(
            util.pytorch_cos_sim(generated_ph_answer_embedding, ph_answer_embedding)
        )
        for false_answer in example["perturbed_answer"]:
            false_answer_embedding = sentencemodel.encode(
                false_answer, convert_to_tensor=True
            )
            scores.append(
                util.pytorch_cos_sim(
                    generated_ph_answer_embedding, false_answer_embedding
                )
            )
        if max(scores) == scores[0]:
            corr += 1
        truth_ratio = (
            sum(false_answers_probs)
            / len(false_answers_probs)
            / (paraphrased_answer_prob+1e-12)
        )
        mean_truth_ratio += truth_ratio
        truth_ratios.append(truth_ratio)
        prompt = example["question"]
        truth_answer = example["answer"]
        truth_answer_prob = compute_prob(
            model, prompt, truth_answer, tokenizer, if_llama=if_llama, if_system=if_system
        )
        mean_truth_prob += truth_answer_prob
        generated_answer = generate_answer(model, tokenizer, prompt, if_llama=if_llama, if_system=if_system)
        score = scorers.score(truth_answer, generated_answer)
        mean_rougeL_score += score["rougeL"].recall
    mean_truth_prob /= len(test_dataset)
    mean_truth_ratio /= len(test_dataset)
    mean_rougeL_score /= len(test_dataset)
    return (
        truth_ratios,
        mean_truth_ratio,
        mean_truth_prob,
        mean_rougeL_score,
        corr / total,
        generated_answers,
    )


def eval_tofu_other(model, tokenizer, subset="retain", if_llama=False,if_system=False):
    dataset = ToFU("TOFU", subset=subset)
    dataset = dataset.build_dataset(tokenizer)
    test_dataset = dataset["test"]
    mean_truth_ratio = 0
    mean_truth_prob = 0
    mean_rougeL_score = 0
    corr = 0
    total = 0
    generated_answers = []
    truth_ratios = []
    scorers = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    for example in tqdm.tqdm(test_dataset, desc=f"evaluating TOFU {subset} dataset"):
        total += 1
        prompt = example["question"]
        false_answers_prob = []
        truth_answer = example["answer"]
        truth_answer_prob = compute_prob(
            model, prompt, truth_answer, tokenizer, if_llama=if_llama, if_system=if_system
        )
        mean_truth_prob += truth_answer_prob
        generated_answer = generate_answer(
            model, tokenizer, prompt, if_llama=if_llama, if_system=if_system
        ).replace("[pad]", "")

        generated_answers.append(generated_answer)
        for false_answer in example["perturbed_answer"]:
            false_answer_prob = compute_prob(
                model, prompt, false_answer, tokenizer, if_llama=if_llama, if_system=if_system
            )
            false_answers_prob.append(false_answer_prob)
        pattern = re.compile(re.escape(truth_answer), re.IGNORECASE)
        if pattern.search(generated_answer) is not None:
            corr += 1
        truth_ratio = (
            sum(false_answers_prob)
            / len(false_answers_prob)
            / (truth_answer_prob+1e-12)
        )
        mean_truth_ratio += truth_ratio
        truth_ratios.append(truth_ratio)
        score = scorers.score(truth_answer, generated_answer)
        mean_rougeL_score += score["rougeL"].recall
    mean_truth_prob /= len(test_dataset)
    mean_truth_ratio /= len(test_dataset)
    mean_rougeL_score /= len(test_dataset)
    return (
        truth_ratios,
        mean_truth_ratio,
        mean_truth_prob,
        mean_rougeL_score,
        corr / total,
        generated_answers,
    )

def infernece(model,tokenizer,text,ex):
    pred = {}
    p1, all_prob, p1_likelihood = calculatePerplexity(text, model, tokenizer)
    p_lower, _, p_lower_likelihood = calculatePerplexity(
        text.lower(), model, tokenizer
    )

    # ppl
    # pred["ppl"] = p1
    # # Ratio of log ppl of lower-case and normal-case
    # # pred["ppl/lowercase_ppl"] = -(np.log(p_lower) / np.log(p1)).item()
    # # Ratio of log ppl of large and zlib
    # zlib_entropy = len(zlib.compress(bytes(text, "utf-8")))
    # pred["ppl/zlib"] = np.log(p1) / zlib_entropy
    # min-k prob
    for ratio in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
        k_length = int(len(all_prob) * ratio)
        topk_prob = np.sort(all_prob)[:k_length]
        pred[f"Min_{ratio*100}% Prob"] = -np.mean(topk_prob).item()
    # print(pred)
    ex["pred"] = pred
    return ex

def MIA(model,tokenizer,forget_subset,remain_subset,if_llama=False,if_system=False):
    question_start_token = "[INST] " if if_llama else "### Question: "
    if if_system:
        question_start_token = "[INST] " + sys_prompt + " " if if_llama else "### Question: " + sys_prompt + " "
    question_end_token = " [/INST]" if if_llama else "\n"
    answer_start_token = " " if if_llama else "### Answer: "
    retain_set = ToFU("TOFU", subset= remain_subset)
    retain_set = retain_set.build_dataset(tokenizer)
    forget_set = ToFU("TOFU", subset= forget_subset)
    forget_set = forget_set.build_dataset(tokenizer)
    real_athours = ToFU("TOFU", subset= "real_authors")
    world_facts = ToFU("TOFU", subset= "world_facts")
    real_athours = real_athours.build_dataset(tokenizer)
    world_facts = world_facts.build_dataset(tokenizer)
    test_set = concatenate_datasets([real_athours["test"],world_facts["test"]])
    dataset = concatenate_datasets([forget_set["test"],test_set])
    labels = []
    for i in range(len(test_set)):
        labels.append(0)
    for i in range(len(forget_set["test"])):
        labels.append(1)
    all_output = []
    for i in tqdm.tqdm(range(len(dataset["question"])), desc="all data"):
        prompt = dataset["question"][i]
        answer = dataset["answer"][i]
        text = question_start_token + prompt + question_end_token + answer_start_token + answer
        ex = {}
        new_ex = infernece(model,tokenizer,text,ex)
        all_output.append(new_ex)
    results = {}
    metric2predictions = defaultdict(list)
    for ex in all_output:
        for metric in ex["pred"].keys():
            metric2predictions[metric].append(ex["pred"][metric])
    for metric, predictions in metric2predictions.items():
        print(len(predictions))
        auc = roc_auc_score(labels, predictions)
        results[metric] = auc
    return results

def eval_tofu(
    model_name,
    forget_subset="forget01",
    retain_subset="retain99",
    output_dir=".",
    if_llama=False,
    if_system=False,
):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        cache_dir="./.cache",
        low_cpu_mem_usage=True,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    left_pad_tokenizer = tokenizer
    left_pad_tokenizer.padding_side = "left"
    left_pad_tokenizer.padding_size = "longest"
    try:
        left_pad_tokenizer.pad_token = left_pad_tokenizer.eos_token
        left_pad_tokenizer.pad_token_id = left_pad_tokenizer.eos_token_id
    except:
        left_pad_tokenizer.pad_token = left_pad_tokenizer.eos_token
        left_pad_tokenizer.pad_token_id = left_pad_tokenizer.eos_token_id
    tokenizer = left_pad_tokenizer
    # acc, rougeL, generated_answers = eval_tofu_adv(
    #     model, tokenizer, subset="forget10",if_llama=if_llama, shots=1
    # )
    # print(acc, rougeL)
    # with open(f"{output_dir}/adv.json", "w") as f:
    #     json.dump({"acc": acc, "rougeL": rougeL, "generated_answers": generated_answers}, f, indent=4)
    AUCs = MIA(model,tokenizer,forget_subset,retain_subset,if_llama=if_llama,if_system=if_system)
    print(AUCs)
    (
        forget_truth_ratios,
        mean_forget_truth_ratio,
        mean_forget_truth_prob,
        mean_forget_rougeL_score,
        mean_forget_acc,
        forget_generated_answers,
        forget_original_answers,
    ) = eval_tofu_forget(model, tokenizer, forget_subset, if_llama=if_llama,if_system=if_system)
    print(mean_forget_acc)
    (
        retain_truth_ratios,
        mean_retain_truth_ratio,
        mean_retain_truth_prob,
        mean_retain_rougeL_score,
        mean_retain_acc,
        retain_generated_answers,
    ) = eval_tofu_retain(model, tokenizer, retain_subset, if_llama=if_llama,if_system=if_system)
    (
        real_author_truth_ratios,
        mean_real_author_truth_ratio,
        mean_real_author_truth_prob,
        mean_real_author_rougeL_score,
        mean_real_author_acc,
        real_author_generated_answers,
    ) = eval_tofu_other(model, tokenizer, "real_authors", if_llama=if_llama,if_system=if_system)
    print(mean_real_author_acc)
    (
        world_fact_truth_ratios,
        mean_world_fact_truth_ratio,
        mean_world_fact_truth_prob,
        mean_world_fact_rougeL_score,
        mean_world_fact_acc,
        world_fact_generated_answers,
    ) = eval_tofu_other(model, tokenizer, "world_facts", if_llama=if_llama,if_system=if_system)
    print(mean_world_fact_acc)

    test_res = ks_2samp(forget_truth_ratios, retain_truth_ratios)
    result = {
        "forget": {
            "truth_ratio": mean_forget_truth_ratio,
            "truth_prob": mean_forget_truth_prob,
            "rougeL_score": mean_forget_rougeL_score,
            "acc": mean_forget_acc,
            "generated_answers": forget_generated_answers,
            "original_answers": forget_original_answers,
        },
        "retain": {
            "truth_ratio": mean_retain_truth_ratio,
            "truth_prob": mean_retain_truth_prob,
            "rougeL_score": mean_retain_rougeL_score,
            "acc": mean_retain_acc,
            "generated_answers": retain_generated_answers,
        },
        "real_author": {
            "truth_ratio": mean_real_author_truth_ratio,
            "truth_prob": mean_real_author_truth_prob,
            "rougeL_score": mean_real_author_rougeL_score,
            "acc": mean_real_author_acc,
            "generated_answers": real_author_generated_answers,
        },
        "world_fact": {
            "truth_ratio": mean_world_fact_truth_ratio,
            "truth_prob": mean_world_fact_truth_prob,
            "rougeL_score": mean_world_fact_rougeL_score,
            "acc": mean_world_fact_acc,
            "generated_answers": world_fact_generated_answers,
        },
        "Forget Quality": test_res.pvalue,
        "MIA": AUCs
    }
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/tofu.json", "w") as f:
        json.dump(result, f, indent=4)

In [ ]:
if __name__ == "__main__":
    model_path = model_path_dir   # your local model folder
    out_dir = EVAL_OUTPUT_DIR

    eval_tofu(
        model_name=model_path,
        forget_subset="forget10",      # common: forget01, forget05, forget10
        retain_subset="retain90",      # common: retain90, retain95, retain99
        output_dir=out_dir,
        if_llama=False,                # set True only if your model expects [INST] ... [/INST]
        if_system=False,               # set True if you want to include sys_prompt in questions
    )


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/3600 [00:00<?, ? examples/s]

retain_perturbed.json: 0.00B [00:00, ?B/s]

...

world_facts_perturbed.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/117 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

all data:   0%|          | 3/617 [00:02<08:45,  1.17it/s]

In [ ]:
import shutil
import os

# Colab output folder
out_dir = EVAL_OUTPUT_DIR

# Where to save zip in Google Drive
drive_zip = DRIVE_ZIP_PATH

# Create zip
shutil.make_archive(
    base_name=drive_zip.replace(".zip", ""),
    format="zip",
    root_dir=out_dir
)

print("✅ Zipped and uploaded to Google Drive:")
print(drive_zip)
